# Fisher consistency test - what's the minimum SNR at which the Fisher formalism is self-consistent?
> From Vallisneri 2008, "Use and abuse of the Fisher information matrix..."
>
> https://arxiv.org/abs/gr-qc/0703086)

> Main script: `production/fisher_consistency_normalized.py`

## 1. How the test works

Fisher forecasts assume a **linear signal model**: $h(\theta+\delta\theta)\approx h(\theta)+\partial_i h\,\delta\theta^i$. Vallisneri’s maximum-mismatch test asks when that assumption is self-consistent on the Fisher $1\sigma$ surface.

- Sample random displacements $\delta\theta$ with $\delta\theta^T C^{-1}\delta\theta=1$.
- Compare the exact waveform $h(\theta+\delta\theta)$ to the linearized waveform.
- Mismatch statistic:
$$|\log r|=\tfrac12\langle h_{\mathrm{lin}}-h_{\mathrm{exact}}\,|\,h_{\mathrm{lin}}-h_{\mathrm{exact}}\rangle.$$
- **Trust criterion:** $\geq 90\%$ of directions have $|\log r|<0.1$. The lowest such network SNR is the “trust SNR”.

## 2. Pre-treating our matrices 
Vallisneri requires that **degenerate / unconstrained directions be removed** before sampling the $1\sigma$ surface. These things are done to the Fisher before it's subject to the test:

- Use correlation matrix (normalized covariance) to avoid unit contamination of the eigenvalues

- Drop (condition on) directions with huge eigenvalues caused by degeneracy. Default: `phase`, `chi1z`, `chi2z`, `tcoal` (aligned-spin / phase / coalescence-time degeneracy), for AGN runs also `dL`.

**Diagnostic (run along with the test) to check whether the matrix has been properly treated:**
In the perturbative regime, $\delta\theta\sim\mathrm{SNR}^{-1}$, so the residual $\delta h \sim \partial_\theta^2h (\delta\theta)^2 \sim \mathrm{SNR}^-1$. Then $|\log r| = 1/2 \langle\delta h|\delta h\rangle$ falls roughly as $\mathrm{SNR}^{-2}.$ A fitted slope near $-2$ is therefore a sanity check that the sampled surface is actually shrinking with SNR; if the slope is much flatter, it implies that there are still unconstrained directions, and any quoted trust SNR is meaningless.

## 3. Lensing model options
- **`generic`:** Phenomenological parameterization (`delta_time`, `relative_distance`, `relative_mass`, `delta_iota`, `delta_phase`, `delta_psi`).
- **`agn`:** AGN parameters (`R_orbit`, `src_pos`, `log10_M_lz`) obtained by a Jacobian transform from the `generic` covariance. **This path is preserved from the SDDR runs but not used in this Fisher consistency test**, since the Jacobian is rank-deficient and produces spurious null directions.
- **`agn_intrinsic`:** Native AGN parameters (`R_orbit`, `src_pos`, `log10_M_lz`), no Jacobian.

## 4. Block options
- **`extra` block:** Lensing parameters only; all binary parameters are presumed known.
- **`full` block:** All free Fisher parameters (after aforementioned pre-treatment).

## 5. Setup of the runs

### Injections

| quantity | typical value |
|---|---|
| $R_{\mathrm{orbit}}$ | $100$ |
| $y$ (Einstein) | $0.5$ |
| $\log_{10} M_{lz}$ | $4.0$ ($M_{lz}=10^{4}\,M_\odot$) |
| $M_c$ | $30$ or $80\,M_\odot$ |
| $\eta$ | $0.24$ |
| $\iota$ | $0.99\,\pi/2$ |
| $\mathrm{phase}$ | $2$ |
| $\chi_{1z}$, $\chi_{2z}$ | $0.3$, $0.5$ |
| $t_{\mathrm{coal}}$ | $0$ |
| $\psi$, $\theta$, $\phi$ | $1$, $1.87$, $2.66$ |
| $d_L$ | $1\,\mathrm{Gpc}$ |

### Settings
| quantity | typical value |
|---|---|
| freq. resolution  | $200$ |
| SNR grid | $\mathrm{geomspace}(3,80,20)$ |
| directions / SNR | $256$ |
| trust cut | $\geq 90{\%}$ with $\|\log r\|<0.1$ |
| condition on | `phase`, `chi1z`, `chi2z`, `tcoal` (+ `dL` for AGN) |
| seed | $42$ |

SNR is changed by rescaling the overall amplitude rather than $d_L$. That keeps the lens angle / redshift geometry fixed when scanning SNR.

## 6. Results

| model | Mc/M_sun | block | trust SNR |
|---|---|---|---|
| generic | 30 | full | 8.7 |
| generic | 30 | extra | 14.2 |
| generic | 80 | full | 16.9 |
| generic | 80 | extra | 89.3 |
| agn_intrinsic | 30 | full | 233 |
| agn_intrinsic | 30 | extra | 580 |
| agn_intrinsic | 80 | full | 520 |
| agn_intrinsic | 80 | extra | >1000 |